In [2]:
import json
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

# ==========================================
# 1.1 LOAD AND PREPROCESS (TEXT DENOISING)
# ==========================================
print("--- 1.1 Loading and Denoising Data ---")
with open('research_abstracts (1).json', 'r', encoding='utf-8') as f:
    raw = json.load(f)

df = pd.DataFrame({
    'DOI': raw['DOI'],
    'Abstract': raw['Abstract'],
    'Label': raw['Label']
})

# Remove missing/irrelevant rows
df = df.dropna(subset=['Label', 'Abstract'])
df = df[df['Abstract'].str.strip().astype(bool)]

def denoise(text: str) -> str:
    if not isinstance(text, str): return ""
    t = text.lower()
    # Remove LaTeX/math
    t = re.sub(r'\$[^$]*\$', ' ', t)
    t = re.sub(r'\\\([^)]*\\\)', ' ', t)
    t = re.sub(r'\[formula:[^\]]*\]', ' ', t)
    t = re.sub(r'\\[a-z]+', ' ', t)
    # Remove URLs/DOIs/emails
    t = re.sub(r'http\S+|www\.\S+', ' ', t)
    t = re.sub(r'10\.\d{4,9}/\S+', ' ', t)
    t = re.sub(r'\S+@\S+', ' ', t)
    # Remove publication boilerplate
    boilerplate = [
        r'copyright\s*©?[^.]*\.', r'this article is part of a special issue[^.]*\.',
        r'institutional subscribers to the nber working paper series[^.]*\.',
        r'residents of developing countries may download[^.]*\.',
        r'jel classification[^.]*\.', r'keywords?:[^.]*\.',
        r'doi:\s*\S+', r'all rights reserved[^.]*\.', r'published by [^.]*\.'
    ]
    for pat in boilerplate:
        t = re.sub(pat, ' ', t)
    # Remove citations and markup
    t = re.sub(r'\[\d+(?:[,\-–]\d+)*\]', ' ', t)
    t = re.sub(r'\(see [^)]*\)', ' ', t)
    t = re.sub(r'[_^~`|<>{}@#$%&*+=]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df['clean_abstract'] = df['Abstract'].apply(denoise)
df = df[df['clean_abstract'].str.len() > 50].copy()

# Tokenization & Lemmatization
stop_words = set(stopwords.words('english')) - {'not', 'no', 'nor', 'never', 'neither'}
lemmatizer = WordNetLemmatizer()

def tokenize_lemma(text):
    tokens = re.findall(r"[a-z]{2,}", text)
    tokens = [w for w in tokens if w not in stop_words]
    return ' '.join([lemmatizer.lemmatize(w) for w in tokens])

df['processed'] = df['clean_abstract'].apply(tokenize_lemma)
df['target'] = (df['Label'] == 'ai').astype(int)
print(f"Data successfully preprocessed. Usable rows: {len(df)}\n")


# ==========================================
# 1.2 FEATURE ENGINEERING
# ==========================================
print("--- 1.2 Feature Engineering (TF-IDF) ---")
# Using unigrams and bigrams to capture phrases like "neural network" or "phillips curve"
tfidf = TfidfVectorizer(
    ngram_range=(1, 2), min_df=3, max_df=0.85, sublinear_tf=True, 
    max_features=8000, strip_accents='unicode'
)
X = tfidf.fit_transform(df['processed'])
y = df['target'].values
print(f"Feature matrix created. Shape: {X.shape}\n")


# ==========================================
# 1.3 BASELINE MODEL & PERFORMANCE REPORT
# ==========================================
print("--- 1.3 Baseline Model Training & Evaluation ---")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

baseline_model = LogisticRegression(
    C=1.0, max_iter=2000, class_weight='balanced', solver='liblinear', random_state=42
)
baseline_model.fit(X_train, y_train)

y_train_pred = baseline_model.predict(X_train)
y_test_pred  = baseline_model.predict(X_test)
y_test_prob  = baseline_model.predict_proba(X_test)[:, 1]

print("\n=== TRAIN METRICS (LEARNING) ===")
print(classification_report(y_train, y_train_pred, target_names=['classical', 'ai']))
print(f"Train Accuracy: {accuracy_score(y_train, y_train_pred):.4f}")

print("\n=== TEST METRICS (GENERALIZATION) ===")
print(classification_report(y_test, y_test_pred, target_names=['classical', 'ai']))
print(f"Test Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Test ROC-AUC : {roc_auc_score(y_test, y_test_prob):.4f}")

print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, y_test_pred))


# ==========================================
# COMMENTARY ON PERFORMANCE
# ==========================================
print("\n" + "="*50)
print("COMMENTARY ON MODEL PERFORMANCE")
print("="*50)
print("""
1. Learning (Training Set): 
The baseline Logistic Regression model achieved a training accuracy of 98.57% and a macro F1-score of 0.98. The near-perfect precision (1.00) for the 'classical' class shows the model easily learned the distinct vocabulary of the training data, meaning it has the capacity to fit the data well without underfitting.

2. Generalization (Test Set): 
The model achieved a test accuracy of 91.67% with a ROC-AUC of 0.956. The drop from training (98.5%) to testing (91.6%) is about 7%, which is a healthy, modest gap. This indicates the model is not severely overfitting, largely thanks to the TF-IDF sublinear scaling and class_weight='balanced' regularization used in the pipeline.

3. Class Imbalance & Error Analysis:
- The model is highly effective at identifying the majority 'classical' class (94% recall).
- The minority 'AI' class has a lower precision (0.78) and recall (0.83), resulting in 8 false negatives and 11 false positives. Because the dataset is heavily imbalanced (roughly 80% classical, 20% AI), the model struggles slightly more with the minority class.
- However, because the class_weight='balanced' parameter was used, the model still caught 40 out of 48 AI papers (83% recall), preventing the model from just guessing 'classical' every time.

Conclusion: 
For a linear baseline model, ~92% accuracy and 0.95 ROC-AUC is a very strong starting point. The linear boundary successfully separates AI methodology vocabulary (e.g., 'neural', 'lstm') from economic vocabulary (e.g., 'phillips curve', 'monetary'). This provides a solid floor to beat in Question 2.
""")

--- 1.1 Loading and Denoising Data ---
Data successfully preprocessed. Usable rows: 1138

--- 1.2 Feature Engineering (TF-IDF) ---
Feature matrix created. Shape: (1138, 8000)

--- 1.3 Baseline Model Training & Evaluation ---

=== TRAIN METRICS (LEARNING) ===
              precision    recall  f1-score   support

   classical       1.00      0.99      0.99       720
          ai       0.95      0.98      0.97       190

    accuracy                           0.99       910
   macro avg       0.97      0.99      0.98       910
weighted avg       0.99      0.99      0.99       910

Train Accuracy: 0.9857

=== TEST METRICS (GENERALIZATION) ===
              precision    recall  f1-score   support

   classical       0.95      0.94      0.95       180
          ai       0.78      0.83      0.81        48

    accuracy                           0.92       228
   macro avg       0.87      0.89      0.88       228
weighted avg       0.92      0.92      0.92       228

Test Accuracy: 0.9167
Tes